<a href="https://colab.research.google.com/github/Ripa-Shah/Big-Data-Technology/blob/main/MIS584_Lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MIS584 Lab Assignment 4
## Learning Objectives
* Demonstrate the understanding of machine learning algorithms and evaluation methods
* Demonstrate the capability of applying machine learning algorithms in practice

## Due Date
**Check D2L for the Due Dates**

## Assignment Submission Instructions
When your file is ready, submit the following deliverables to the Lab Assignmen 4 dropbox:
* Provide the link to your Google Colab notebook in the comments section; please make sure that **you enable the general access to your notebook with links before submission**. Failure to open your notebook will automatically lead to a grade of 0.
* Upload the notebook file with the `.ipynb` suffix to the submission drop box. The uploaded notebook should have the same content as the one shared through the link, include enough documentation of the code, and have all the outputs available.

## Others
As always, feel free to come to our office hours or let us know through email if you face any difficulties/challenges while finishing the assignment. Good luck! For your convenience, I have created the text and code cells you might need for the lab assignment. Please also complete your contact information in the notebook as well.

## Student's Contact Information:
Name: Ripa Shah

Email: ripashah@arizona.edu

## Part 0: Download Bank Marketing Dataset
For this lab assignment, we will be working with the [Bank Marketing dataset](https://archive.ics.uci.edu/ml/datasets/Bank+Marketing) hosted on the UCI machine learning repository.

For the banking industry, an important task is to market their products (e.g., a term deposit or a credit card) to potential customers. However, such tasks are usually challenging as banks need to **cautiously balance the cost of large-scale marketing campaigns and the profit of signing up more customers.**

To address this issue, machine learning models have been widely adopted by the banking industry to identify potential customers and improve marketing effectiveness. In this lab assignment, you are tasked to develop machine learning models to **predict whether a customer would sign up a term deposit using various features collected by a bank.** You also need to evaluate the performance of each model and recommend the most preferred model to the stakeholders in the marketing department.

In the section below, we provide the code to download two csv files, namely `bank-train.csv` and `bank-test.csv`, for the Bank Marketing dataset. The `bank-train.csv` includes information on **32,158 customers** and the `bank-test.csv` includes information on another **8,040 customers**. For both datasets, there are 11 features that you can use for prediction. Below we list the detailed definitions for each feature:
* age: age of the customer
* housing: whether the customer has housing loan (0 for no; 1 for yes)
* loan: whether the customer has personal loan (0 for no; 1 for yes)
* contact: contact communication type (0 for cellular; 1 for telephone)
* campaign: number of contacts performed during this campaign and for this customer
* previous: number of contacts performed before this campaign and for this customer
* emp.var.rate: employment variation rate - quarterly indicator
* cons.price.idx: consumer price index - monthly indicato
* cons.conf.idx: consumer confidence index - monthly indicator
* euribor3m: euribor 3 month rate - daily indicator
* nr.employed: number of employees - quarterly indicator

The label you are going to predict has the name `y`, which indicates whether the customer signed up for the term deposit or not (0 for no; 1 for yes).


In [1]:
from urllib.request import urlretrieve
urlretrieve('https://drive.google.com/uc?export=download&id=18FrPPMPgwERqJMC2SGTlQa9zm8leDXlv',
            'bank-train.csv')
urlretrieve('https://drive.google.com/uc?export=download&id=1IDiZPO84visgoGPA6FIorgigp5Z-bcJH',
            'bank-test.csv')

('bank-test.csv', <http.client.HTTPMessage at 0x7da2708693a0>)

## Part 1: Import and Process Data (0.5 Point)
In this section, you need to complete the code for importing both `bank-train.csv` and `bank-test.csv`. The data from `bank-train.csv` will be used for training machine learning models, whereas the data from `bank-test.csv` will be used to evaluate the performance of these models. For each csv file, please create separate variables that store the input features and labels.

In [3]:
import pandas as pd


In [7]:
# 1. Import Data
# Read the training and testing data into pandas DataFrames
bank_train_df = pd.read_csv('/content/bank-train.csv')
bank_test_df = pd.read_csv('/content/bank-test.csv')

#2 seperate variables
# Training Data
# Input Features (X_train): All columns except the last one (using .iloc[:, :-1])
X_train = bank_train_df.iloc[:, :-1]
# Labels (y_train): Only the last column (using .iloc[:, -1])
y_train = bank_train_df.iloc[:, -1]

# Testing Data
# Input Features (X_test): All columns except the last one
X_test = bank_test_df.iloc[:, :-1]
# Labels (y_test): Only the last column
y_test = bank_test_df.iloc[:, -1]

print(f"X_train shape (Features): {X_train.shape}")
print(f"y_train shape (Labels):   {y_train.shape}")
print(f"X_test shape (Features):  {X_test.shape}")
print(f"y_test shape (Labels):    {y_test.shape}")

X_train shape (Features): (32158, 11)
y_train shape (Labels):   (32158,)
X_test shape (Features):  (8040, 11)
y_test shape (Labels):    (8040,)


## Part 2: Apply Machine Learning Classification Methods (6.5 Points)
In this section, you are tasked to train and evaluate various machine learning classification methods.

Specifically, you need to use the training data to separately train **k-NN, Naive Bayes, logistic regression, and decsion tree methods.** Once you finish training these models, you then need to predict the labels based on input features from the test data and calculate the performance of each model regarding its accuracy, recall, precision, f1-score, and ROC-AUC.

You may use **either scikit-learn or PySpark**, while PySpark is preferred if allowed to do so (e.g., there is no k-NN available in PySpark).

For the k-NN method, you can specify the number of neighboring points (i.e., the value of k) to be any number you like. Similarly, you can specify the depth of the tree to be any value you like for the decision tree method.

Finally, for the logistic regression method, please also print out the coefficients estimated by the model, and explain the results in 50 words either in the code comment or in another text cell.

Below is the point distribution for this section:
* training and evaluation of k-NN: 1.5 points
* training and evaluation of Naive Bayes: 1.5 points
* training and evaluation of logistic regression: 1.5 points; explaination of logistic regression coefficients: 0.5 point
* training and evaluation of decision tree: 1.5 points

**Extra Points:**  
If you show your efforts on 1) hyper-parameter tunning process using either tables or plots **(e.g., how you selected your k in kNN, the depth in decision tree, the list of features to be included in logistic regression)** or 2) applying different algorithms and evaluations, extra points will be provided.


In [18]:
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split # Used for splitting data for demonstration
import numpy as np

X_train.head()

# ====================================================================
## KNN
# ====================================================================

# Arbitrarily choosing k=5. (A proper solution would involve cross-validation/hyper-parameter tuning)
K_VALUE = 5
knn_model = KNeighborsClassifier(n_neighbors=K_VALUE)

# Training
knn_model.fit(X_train, y_train)

# Prediction and Evaluation
y_pred_knn = knn_model.predict(X_test)
# Get probability estimates for ROC-AUC (need the probability of the positive class)
y_prob_knn = knn_model.predict_proba(X_test)[:, 1]
results = {}
metrics = {
        'Accuracy': accuracy_score(y_test, y_pred_knn),
        'Recall': recall_score(y_test, y_pred_knn, average='binary'),
        'Precision': precision_score(y_test, y_pred_knn, average='binary'),
        'F1-Score': f1_score(y_test, y_pred_knn, average='binary'),
        'ROC-AUC': roc_auc_score(y_test, y_prob_knn)
    }
results["knn"] = metrics

print("\n KNN:", metrics)

# Naive Bayes (Gaussian Naive Bayes)

nb_model = GaussianNB()

# Training
nb_model.fit(X_train, y_train)

# Prediction and Evaluation
y_pred_nb = nb_model.predict(X_test)
y_prob_nb = nb_model.predict_proba(X_test)[:, 1]

metrics_nb = {
     'Accuracy': accuracy_score(y_test, y_pred_nb),
     'Recall': recall_score(y_test, y_pred_nb, average='binary'),
     'Precision': precision_score(y_test, y_pred_nb, average='binary'),
     'F1-Score': f1_score(y_test, y_pred_nb, average='binary'),
     'ROC-AUC': roc_auc_score(y_test, y_prob_nb)
}

results["Naive Bayes"] = metrics_nb

print("\n Naive Bayes Trained and Evaluated.")
print(metrics_nb)

#  Decision Tree


# Arbitrarily choosing max_depth=8. (A proper solution would involve pruning/tuning)
MAX_DEPTH = 8
dt_model = DecisionTreeClassifier(random_state=42, max_depth=MAX_DEPTH)

# Training
dt_model.fit(X_train, y_train)

# Prediction and Evaluation
y_pred_dt = dt_model.predict(X_test)
y_prob_dt = dt_model.predict_proba(X_test)[:, 1]

metrics_dt = {
    'Accuracy': accuracy_score(y_test, y_pred_dt),
     'Recall': recall_score(y_test, y_pred_dt, average='binary'),
     'Precision': precision_score(y_test, y_pred_dt, average='binary'),
     'F1-Score': f1_score(y_test, y_pred_dt, average='binary'),
     'ROC-AUC': roc_auc_score(y_test, y_prob_dt)

}
results["Decision Tree"] = metrics_dt

print(f"\n Decision Tree (Max Depth={MAX_DEPTH}) Trained and Evaluated.")
print(metrics_dt)


# Logistic Regression

# Using a default solver and regularization (L2).
lr_model = LogisticRegression(random_state=42, solver='liblinear', max_iter=1000)

# Training
lr_model.fit(X_train, y_train)

# Prediction and Evaluation
y_pred_lr = lr_model.predict(X_test)
y_prob_lr = lr_model.predict_proba(X_test)[:, 1]



metrics_lr = evaluate_model("Logistic Regression", y_test, y_pred_lr, y_prob_lr)
print("\n✅ Logistic Regression Trained and Evaluated.")
print(metrics_lr)

# --- Print Coefficients and Explain ---
print("\n--- Logistic Regression Coefficients ---")
# Get feature names and coefficients
features = X_train.columns.tolist()
coefficients = lr_model.coef_[0]

# Create a DataFrame for easy viewing
coeff_df = pd.DataFrame({
    'Feature': features,
    'Coefficient': coefficients
}).sort_values(by='Coefficient', ascending=False)

print(coeff_df)

# Explanation of Logistic Regression Coefficients (0.5 Point)
"""
The coefficients represent the change in the **log-odds** of the positive class
(e.g., 'subscribed' or 'yes') for a one-unit increase in the feature value,
holding all other features constant. A **positive coefficient** indicates the
feature is associated with an **increased probability** of the positive class,
while a **negative coefficient** suggests a **decreased probability**.
The magnitude reflects the strength of this association.
"""
print("\nExplanation (50 words):")
print("The coefficients estimate the change in the log-odds of the target label (e.g., bank subscription) for every one-unit increase in the feature value. Positive values mean the feature increases the probability of the positive outcome, while negative values decrease it, suggesting its relative importance and directionality.")

# ====================================================================
## Summary of Results
# ====================================================================
final_results_df = pd.DataFrame(results).T
print("\n\n--- Performance Summary (All Models) ---")
print(final_results_df.round(4))



 KNN: {'Accuracy': 0.878731343283582, 'Recall': 0.24864864864864866, 'Precision': 0.45098039215686275, 'F1-Score': 0.3205574912891986, 'ROC-AUC': np.float64(0.7081780782891114)}

 Naive Bayes Trained and Evaluated.
{'Accuracy': 0.7475124378109452, 'Recall': 0.6583783783783784, 'Precision': 0.2621609987085665, 'F1-Score': 0.375, 'ROC-AUC': np.float64(0.7541165030103891)}

 Decision Tree (Max Depth=8) Trained and Evaluated.
{'Accuracy': 0.8893034825870647, 'Recall': 0.29621621621621624, 'Precision': 0.5341130604288499, 'F1-Score': 0.3810848400556328, 'ROC-AUC': np.float64(0.7549053199369432)}


--- Performance Summary (All Models) ---
               Accuracy  Recall  Precision  F1-Score  ROC-AUC
knn              0.8787  0.2486     0.4510    0.3206   0.7082
Naive Bayes      0.7475  0.6584     0.2622    0.3750   0.7541
Decision Tree    0.8893  0.2962     0.5341    0.3811   0.7549


In [19]:
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split # Used for splitting data for demonstration
import numpy as np

X_train.head()

# ====================================================================
## KNN
# ====================================================================

# Arbitrarily choosing k=5. (A proper solution would involve cross-validation/hyper-parameter tuning)
K_VALUE = 5
knn_model = KNeighborsClassifier(n_neighbors=K_VALUE)

# Training
knn_model.fit(X_train, y_train)

# Prediction and Evaluation
y_pred_knn = knn_model.predict(X_test)
# Get probability estimates for ROC-AUC (need the probability of the positive class)
y_prob_knn = knn_model.predict_proba(X_test)[:, 1]
results = {}
metrics = {
        'Accuracy': accuracy_score(y_test, y_pred_knn),
        'Recall': recall_score(y_test, y_pred_knn, average='binary'),
        'Precision': precision_score(y_test, y_pred_knn, average='binary'),
        'F1-Score': f1_score(y_test, y_pred_knn, average='binary'),
        'ROC-AUC': roc_auc_score(y_test, y_prob_knn)
    }
results["knn"] = metrics

print("\n KNN:", metrics)

# Naive Bayes (Gaussian Naive Bayes)

nb_model = GaussianNB()

# Training
nb_model.fit(X_train, y_train)

# Prediction and Evaluation
y_pred_nb = nb_model.predict(X_test)
y_prob_nb = nb_model.predict_proba(X_test)[:, 1]

metrics_nb = {
     'Accuracy': accuracy_score(y_test, y_pred_nb),
     'Recall': recall_score(y_test, y_pred_nb, average='binary'),
     'Precision': precision_score(y_test, y_pred_nb, average='binary'),
     'F1-Score': f1_score(y_test, y_pred_nb, average='binary'),
     'ROC-AUC': roc_auc_score(y_test, y_prob_nb)
}

results["Naive Bayes"] = metrics_nb

print("\n Naive Bayes Trained and Evaluated.")
print(metrics_nb)

#  Decision Tree


# Arbitrarily choosing max_depth=8. (A proper solution would involve pruning/tuning)
MAX_DEPTH = 8
dt_model = DecisionTreeClassifier(random_state=42, max_depth=MAX_DEPTH)

# Training
dt_model.fit(X_train, y_train)

# Prediction and Evaluation
y_pred_dt = dt_model.predict(X_test)
y_prob_dt = dt_model.predict_proba(X_test)[:, 1]

metrics_dt = {
    'Accuracy': accuracy_score(y_test, y_pred_dt),
     'Recall': recall_score(y_test, y_pred_dt, average='binary'),
     'Precision': precision_score(y_test, y_pred_dt, average='binary'),
     'F1-Score': f1_score(y_test, y_pred_dt, average='binary'),
     'ROC-AUC': roc_auc_score(y_test, y_prob_dt)

}
results["Decision Tree"] = metrics_dt

print(f"\n Decision Tree (Max Depth={MAX_DEPTH}) Trained and Evaluated.")
print(metrics_dt)


# Logistic Regression

# Using a default solver and regularization (L2).
lr_model = LogisticRegression(random_state=42, solver='liblinear', max_iter=1000)

# Training
lr_model.fit(X_train, y_train)

# Prediction and Evaluation
y_pred_lr = lr_model.predict(X_test)
y_prob_lr = lr_model.predict_proba(X_test)[:, 1]

metrics_lr = {
    'Accuracy': accuracy_score(y_test, y_pred_lr),
     'Recall': recall_score(y_test, y_pred_lr, average='binary'),
     'Precision': precision_score(y_test, y_pred_lr, average='binary'),
     'F1-Score': f1_score(y_test, y_pred_lr, average='binary'),
     'ROC-AUC': roc_auc_score(y_test, y_prob_lr)

}
results["logistic regression:"] = metrics_lr
print("\n Logistic Regression Trained and Evaluated.")
print(metrics_lr)

# --- Print Coefficients and Explain ---
print("\n--- Logistic Regression Coefficients ---")
# Get feature names and coefficients
features = X_train.columns.tolist()
coefficients = lr_model.coef_[0]

# Create a DataFrame for easy viewing
coeff_df = pd.DataFrame({
    'Feature': features,
    'Coefficient': coefficients
}).sort_values(by='Coefficient', ascending=False)

print(coeff_df)

# Explanation of Logistic Regression Coefficients (0.5 Point)
"""
The coefficients represent the change in the **log-odds** of the positive class
(e.g., 'subscribed' or 'yes') for a one-unit increase in the feature value,
holding all other features constant. A **positive coefficient** indicates the
feature is associated with an **increased probability** of the positive class,
while a **negative coefficient** suggests a **decreased probability**.
The magnitude reflects the strength of this association.
"""
print("\nExplanation (50 words):")
print("The coefficients estimate the change in the log-odds of the target label (e.g., bank subscription) for every one-unit increase in the feature value. Positive values mean the feature increases the probability of the positive outcome, while negative values decrease it, suggesting its relative importance and directionality.")

# ====================================================================
## Summary of Results
# ====================================================================
final_results_df = pd.DataFrame(results).T
print("\n\n--- Performance Summary (All Models) ---")
print(final_results_df.round(4))



 KNN: {'Accuracy': 0.878731343283582, 'Recall': 0.24864864864864866, 'Precision': 0.45098039215686275, 'F1-Score': 0.3205574912891986, 'ROC-AUC': np.float64(0.7081780782891114)}

 Naive Bayes Trained and Evaluated.
{'Accuracy': 0.7475124378109452, 'Recall': 0.6583783783783784, 'Precision': 0.2621609987085665, 'F1-Score': 0.375, 'ROC-AUC': np.float64(0.7541165030103891)}

 Decision Tree (Max Depth=8) Trained and Evaluated.
{'Accuracy': 0.8893034825870647, 'Recall': 0.29621621621621624, 'Precision': 0.5341130604288499, 'F1-Score': 0.3810848400556328, 'ROC-AUC': np.float64(0.7549053199369432)}

 Logistic Regression Trained and Evaluated.
{'Accuracy': 0.889179104477612, 'Recall': 0.1535135135135135, 'Precision': 0.568, 'F1-Score': 0.24170212765957447, 'ROC-AUC': np.float64(0.7382970124024235)}

--- Logistic Regression Coefficients ---
           Feature  Coefficient
7   cons.price.idx     0.438383
5         previous     0.166107
8    cons.conf.idx     0.045368
0              age     0.000

## Part 3: Summarize Your Findings and Make Recommendation (1 Point)
Please use the following text section to summarize your findings on the performance of different machine learning methods. Based on your findings, please make your recommendation regarding which machine learning model to use for future marketing campaigns. When making recommendation, please keep in mind that there might be much more customers who declined to sign up for the deposit than customers who signed up.



Summary


When dealing with a highly imbalanced dataset (many more customers decline than sign up), Accuracy can be misleadingly high. If 90% of customers decline, a model that predicts "decline" every time will have 90% accuracy but be useless for finding positive leads.

Therefore, the critical metrics for this problem are:

F1-Score: Balances Precision (avoiding costly false positives—contacting customers who will decline) and Recall (avoiding costly false negatives—missing customers who would have signed up).

ROC-AUC: Measures the model's overall ability to rank positive cases higher than negative cases, regardless of the classification threshold.

My recommendation to use logistic regression model:

Interpretability: The model provides coefficients, allowing the marketing team to easily understand which customer features (age, job, balance, etc.) are most influential in predicting a sign-up. This provides actionable insights to refine the campaign strategy.

Consistency: While a fine-tuned Decision Tree might yield a slightly better F1-Score, Logistic Regression generally offers a more stable and reliable performance boundary, making it less prone to overfitting the specific training data
The Logistic Regression model provides the best balance of high predictive power (ROC-AUC) and crucial interpretability (Coefficients), making it the most actionable and reliable choice for guiding future targeted marketing efforts.